In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

In [2]:
# %%capture --no-stderr
# !pip install python-dotenv langchain-community langchain-core langchain langchain-openai langchain-chroma

In [3]:
# 환경변수 설정

In [4]:
# 라이브러리 불러오기
from dotenv import load_dotenv
import os
from langchain_openai import OpenAI

In [5]:
# .env 파일에서 환경 변수 로드 (.env 파일에는 OPENAI API 키값을 적으면 됩니다. -> OPENAI_API_KEY=...)
load_dotenv("/content/.env")
# 환경 변수에서 API 키 가져오기
api_key = os.getenv("OPENAI_API_KEY")
# 오픈AI 대규모 언어 모델 초기화
llm = OpenAI()

In [6]:
# <이전 대화를 포함한 메시지 전달>

# 라이브러리 불러오기
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

chat = ChatOpenAI(model="gpt-4o-mini")

# 프롬프트 템플릿 정의: 금융 상담 역할
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 금융 상담사입니다. 사용자에게 최선의 금융 조언을 제공합니다."),
        ("placeholder", "{messages}"),  # 대화 이력 추가
    ]
)

# 프롬프트와 모델을 연결하여 체인 생성
chain = prompt | chat

# 이전 대화를 포함한 메시지 전달
ai_msg = chain.invoke(
    {
        "messages": [
            ("human", "저축을 늘리기 위해 무엇을 할 수 있나요?"),  # 사용자의 첫 질문
            ("ai", "저축 목표를 설정하고, 매달 자동 이체로 일정 금액을 저축하세요."),  # 챗봇의 답변
            ("human", "방금 뭐라고 했나요?"),  # 사용자의 재확인 질문
        ],
    }
)
print(ai_msg.content)  # 챗봇의 응답 출력


저축을 늘리기 위해 목표를 설정하고, 매달 자동 이체를 통해 일정 금액을 저축하라고 말씀드렸습니다. 이렇게 하면 저축 습관을 형성하고, 정기적으로 저축할 수 있습니다. 추가적으로, 소비를 줄이고 예산을 잘 관리하는 것도 도움이 됩니다.


In [7]:
# <`ChatMessageHistory`를 사용한 메시지 관리>

from langchain_community.chat_message_histories import ChatMessageHistory

# 대화 이력 저장을 위한 클래스 초기화
chat_history = ChatMessageHistory()

# 사용자 메시지 추가
chat_history.add_user_message("저축을 늘리기 위해 무엇을 할 수 있나요?")
chat_history.add_ai_message("저축 목표를 설정하고, 매달 자동 이체로 일정 금액을 저축하세요.")

# 새로운 질문 추가 후 다시 체인 실행
chat_history.add_user_message("방금 뭐라고 했나요?")
ai_response = chain.invoke({"messages": chat_history.messages})
print(ai_response.content)  # 챗봇은 이전 메시지를 기억하여 답변합니다.

저축을 늘리기 위해 저축 목표를 설정하고, 매달 자동 이체로 일정 금액을 저축하는 것이 좋다고 말씀드렸습니다. 이렇게 하면 저축 습관을 기르고 쉽게 저축할 수 있습니다. 추가로, 지출을 줄이는 방법이나 예산을 세우는 것도 도움이 될 수 있습니다. 더 궁금한 점이 있으면 말씀해 주세요!


In [8]:
# <` RunnableWithMessageHistory`를 사용한 메시지 관리>

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

# 시스템 메시지와 대화 이력을 사용하는 프롬프트 템플릿 정의
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 금융 상담사입니다. 모든 질문에 최선을 다해 답변하십시오."),
        ("placeholder", "{chat_history}"),  # 이전 대화 이력
        ("human", "{input}"),  # 사용자의 새로운 질문
    ]
)

# 대화 이력을 관리할 체인 설정
chat_history = ChatMessageHistory()
chain = prompt | chat

# RunnableWithMessageHistory 클래스를 사용해 체인을 감쌉니다
chain_with_message_history = RunnableWithMessageHistory(
    chain,
    lambda session_id: chat_history,  # 세션 ID에 따라 대화 이력을 불러오는 함수
    input_messages_key="input",  # 입력 메시지의 키 설정
    history_messages_key="chat_history",  # 대화 이력의 키 설정
)

# RunnableWithMessageHistory 클래스를 사용해 체인을 감쌉니다
chain_with_message_history = RunnableWithMessageHistory(
    chain,
    lambda session_id: chat_history,  # 세션 ID에 따라 대화 이력을 불러오는 함수
    input_messages_key="input",  # 입력 메시지의 키 설정
    history_messages_key="chat_history",  # 대화 이력의 키 설정
)

# 질문 메시지 체인 실행
chain_with_message_history.invoke(
    {"input": "저축을 늘리기 위해 무엇을 할 수 있나요?"},
    {"configurable": {"session_id": "unused"}},
).content


Parent run 08c51736-322b-44d3-a681-9fd5106ff02e not found for run e3d2a7a7-515f-4265-82d2-ad2223277c88. Treating as a root run.


'저축을 늘리기 위해 다음과 같은 몇 가지 방법을 고려할 수 있습니다:\n\n1. **예산 수립**: 월간 수입과 지출을 분석하여 예산을 세웁니다. 필요한 지출과 불필요한 지출을 구분하여 저축할 수 있는 금액을 명확히 합니다.\n\n2. **자동 저축 설정**: 급여가 들어오면 자동으로 저축 계좌로 일정 금액을 이체하도록 설정합니다. 이렇게 하면 저축을 쉽게 시작할 수 있습니다.\n\n3. **비상금 마련**: 비상금을 따로 마련해 두면 예상치 못한 지출로 인해 저축을 줄이는 일을 방지할 수 있습니다. 일반적으로 3~6개월치 생활비를 비상금으로 권장합니다.\n\n4. **비용 절감**: 불필요한 구독 서비스나 고정비용을 줄이는 방법을 찾아보세요. 예를 들어, 불필요한 외식이나 쇼핑을 줄이는 것이 좋습니다.\n\n5. **소득 증대**: 부업이나 자유직업을 통해 추가 소득을 창출할 수 있는 방법을 모색해 보세요. 추가 수입은 저축에 직접적으로 도움이 됩니다.\n\n6. **금융 상품 활용**: 높은 이자율을 제공하는 저축 계좌나 적금 상품을 활용하여 저축의 이자 수익을 극대화할 수 있습니다.\n\n7. **목표 설정**: 구체적인 저축 목표를 설정하면 동기부여가 됩니다. 예를 들어, 특정한 여행이나 대출 상환 등을 목표로 삼을 수 있습니다.\n\n8. **소비 패턴 점검**: 소비 습관을 점검하고, 필요 없는 지출을 줄이기 위한 노력을 기울입니다. 예를 들어, 대중교통 이용, 세일 기간 활용 등이 있습니다.\n\n이런 방법들을 통해 저축을 늘릴 수 있는 기회를 찾아보세요. 중요한 것은 작은 변화들도 꾸준히 실천하는 것입니다.'

In [9]:
# 새로운 입력 메시지를 추가하고 체인을 실행
chain_with_message_history.invoke(
    {"input": "내가 방금 뭐라고 했나요?"},  # 사용자의 질문
    {"configurable": {"session_id": "unused"}}  # 세션 ID 설정
).content


Parent run d45c68cb-edda-49da-ad97-1931875ee2ee not found for run e17a9f97-9d77-4aac-8a74-6521894909b7. Treating as a root run.


'당신은 "저축을 늘리기 위해 무엇을 할 수 있나요?"라고 물으셨습니다. 저축을 늘리기 위한 다양한 방법을 제안해 드렸습니다. 추가적인 질문이나 더 알고 싶은 내용이 있다면 말씀해 주세요!'

In [10]:
# <메시지 트리밍 예제>

# 라이브러리 불러오기
from langchain_core.messages import trim_messages
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter

# 메시지 트리밍 유틸리티 설정
trimmer = trim_messages(strategy="last", max_tokens=2, token_counter=len)

# 트리밍된 대화 이력과 함께 체인 실행
chain_with_trimming = (
    RunnablePassthrough.assign(chat_history=itemgetter("chat_history") | trimmer)
    | prompt
    | chat
)

# 트리밍된 대화 이력을 사용하는 체인 설정
chain_with_trimmed_history = RunnableWithMessageHistory(
    chain_with_trimming,
    lambda session_id: chat_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)

# 새로운 대화 내용 추가 후 체인 실행
chain_with_trimmed_history.invoke(
    {"input": "저는 5년 내에 집을 사기 위해 어떤 재정 계획을 세워야 하나요?"},  # 사용자의 질문
    {"configurable": {"session_id": "finance_session_1"}}  # 세션 ID 설정
)



ImportError: cannot import name 'trim_messages' from 'langchain_core.messages' (d:\WorkSpace\Python\langchain-tutorial\Ch01. Langchain Basics\venv\lib\site-packages\langchain_core\messages\__init__.py)

In [ ]:
# 새로운 입력 메시지를 추가하고 체인을 실행
chain_with_trimmed_history.invoke(
    {"input": "내가 방금 뭐라고 했나요?"},  # 사용자의 질문
    {"configurable": {"session_id": "finance_session_1"}}  # 세션 ID 설정
).content


'당신은 "저는 5년 내에 집을 사기 위해 어떤 재정 계획을 세워야 하나요?" 라고 질문했습니다. 이 질문에 대해 재정 계획을 세우는 단계와 방법을 안내해 드렸습니다. 추가적인 도움이 필요하시면 언제든지 말씀해 주세요!'

In [ ]:
# <이전 대화 요약 내용 기반으로 답변하기>

def summarize_messages(chain_input):
    stored_messages = chat_history.messages
    if len(stored_messages) == 0:
        return False
    # 대화를 요약하기 위한 프롬프트 템플릿 설정
    summarization_prompt = ChatPromptTemplate.from_messages(
        [
            ("placeholder", "{chat_history}"),  # 이전 대화 이력
            (
                "user",
                "이전 대화를 요약해 주세요. 가능한 한 많은 세부 정보를 포함하십시오.",  # 요약 요청 메시지
            ),
        ]
    )

    # 요약 체인 생성 및 실행
    summarization_chain = summarization_prompt | chat
    summary_message = summarization_chain.invoke({"chat_history": stored_messages})
    chat_history.clear()  # 요약 후 이전 대화 삭제
    chat_history.add_message(summary_message)  # 요약된 메시지를 대화 이력에 추가
    return True

In [ ]:
# 대화 요약을 처리하는 체인 설정
chain_with_summarization = (
    RunnablePassthrough.assign(messages_summarized=summarize_messages)
    | chain_with_message_history
)

# 요약된 대화를 기반으로 새로운 질문에 응답
print(chain_with_summarization.invoke(
    {"input": "저에게 어떤 재정적 조언을 해주셨나요?"},  # 사용자의 질문
    {"configurable": {"session_id": "unused"}}  # 세션 ID 설정
).content)


저는 다음과 같은 재정적 조언을 드렸습니다:

1. **저축을 늘리기 위한 방법**:
   - **예산 작성**: 수입과 지출을 기록하여 지출을 관리하고 필요 없는 지출을 줄이는 방법.
   - **자동 저축 설정**: 급여가 들어오면 자동으로 저축 계좌에 송금하는 방식을 통해 저축을 습관화.
   - **소비 습관 점검**: 외식이나 과도한 소액 지출을 줄이는 방법.
   - **비상금 마련**: 예기치 않은 상황에 대비하기 위해 비상금 통장을 만드는 조언.
   - **목표 설정**: 구체적인 저축 목표를 가지고 동기를 부여하는 방식.
   - **고수익 저축계좌 활용**: 일반 저축계좌보다 이자가 높은 상품 활용.
   - **할인과 혜택 활용**: 쇼핑 시 할인 쿠폰과 포인트 적립 프로그램을 적극 활용.
   - **부가 수익 활동 고려**: 아르바이트나 프리랜서 일을 통해 추가 수익을 창출.
   - **정기적인 점검**: 저축 목표를 주기적으로 점검하고 필요 시 조정.
   - **금융 교육**: 재정 관리에 대한 지식을 늘리기 위한 독서 및 강의 수강.

2. **5년 내에 집을 사기 위한 재정 계획**:
   - **목표 설정**: 원하는 집의 가격대를 파악하고 필요한 다운페이먼트를 계산.
   - **예산 수립**: 현재 소득과 지출을 기반으로 예산을 세우고 불필요한 지출을 조정하는 방법.
   - **저축 목표**: 필요한 다운페이먼트를 5년으로 나누어 매달 저축해야 할 금액을 계산.
   - **이자-bearing Account**: 고수익 저축계좌나 투자상품을 통해 자산을 증대시키는 방법.
   - **신용 점수 관리**: 신용 점수를 점검하고 개선하는 것을 통해 대출 조건을 유리하게 만드는 방법.
   - **재정 전문가와 상담**: 필요 시 전문가의 조언을 받는 것.
   - **위험 관리**: 안정적인 투자로 저축하고 다양한 자산에 분산 투자하는 방법.

이 조언들이 도움이 되기를 바랍니다. 추가 질문이나 더 구체적인 조언이 필요하시